In [ ]:
from pathlib import Path
import json, math, re
import numpy as np
import pandas as pd

# ============================ SETTINGS ============================
DRIVE_ROOT = Path('/content/drive/MyDrive')
PREFERRED_ROOTS = [
    DRIVE_ROOT / 'grokking_rebuttal_addition',
    DRIVE_ROOT / 'grokking_rebuttal_s5',
    DRIVE_ROOT / 'grokking_rebuttal_s5_corrected',
]
EXTRA_PATHS = []
OUTPUT_DIR = DRIVE_ROOT / 'grokking_rebuttal_master_summary'
THRESHOLDS = (0.95, 0.99, 1.00)

ADDITION_CONFIG_ORDER = [
    'diag_layernorm_raw_lr_6e-4_wd0_long',
    'diag_layernorm_bounded_lr_6e-4_wd0_long',
    'diag_layernorm_raw_lr_6e-4_long',
    'layernorm_bounded_lr_0.0006_long',
    'diag_sphere_raw_lr_6e-4_wd0_long',
    'diag_sphere_bounded_lr_6e-4_wd0_long',
    'diag_sphere_raw_lr_6e-4_wd1_long',
    'diag_sphere_bounded_lr_6e-4',
]
S5_TERMS = ('s5', 'permutation')
CORRECTED_TERMS = ('corrected', 'posthocfix', 'posthoc_fix', 'sphere_wd1_corrected')
PATH_HINTS = ('grok', 'addition', 's5', 'sphere', 'layernorm', 'result', 'transition', 'summary', 'sweep')

# Onset = first logged evaluation epoch at or above threshold.
# Gap = test onset - train onset at the same threshold.


def first_present(d, keys, default=None):
    for k in keys:
        if k in d and d[k] is not None:
            return d[k]
    return default


def as_float(x):
    try:
        y = float(x)
        return y if math.isfinite(y) else np.nan
    except (TypeError, ValueError):
        return np.nan


def acc(x):
    x = as_float(x)
    if pd.isna(x):
        return np.nan
    return x / 100.0 if x > 1.5 else x


def as_bool(x):
    if isinstance(x, bool):
        return x
    if x is None:
        return np.nan
    t = str(x).strip().lower()
    if t in {'true', '1', 'yes', 'y'}:
        return True
    if t in {'false', '0', 'no', 'n'}:
        return False
    return np.nan


def extract_history(rec):
    for h in (rec.get('history'), rec.get('metrics'), rec.get('log'), rec.get('training_history')):
        if isinstance(h, list) and h:
            rows = []
            for item in h:
                if not isinstance(item, dict):
                    continue
                epoch = first_present(item, ('epoch', 'step', 'iteration'))
                if epoch is None:
                    continue
                rows.append({
                    'epoch': int(float(epoch)),
                    'train_acc': acc(first_present(item, ('train_acc', 'train_accuracy', 'acc_train'))),
                    'test_acc': acc(first_present(item, ('test_acc', 'test_accuracy', 'val_acc', 'validation_accuracy', 'acc_test'))),
                })
            return sorted(rows, key=lambda r: r['epoch'])
        if isinstance(h, dict):
            e = first_present(h, ('epoch', 'epochs', 'step', 'steps'))
            tr = first_present(h, ('train_acc', 'train_accuracy', 'training_accuracy'))
            te = first_present(h, ('test_acc', 'test_accuracy', 'val_acc', 'validation_accuracy'))
            if e is not None and tr is not None and te is not None:
                return sorted([
                    {'epoch': int(float(a)), 'train_acc': acc(b), 'test_acc': acc(c)}
                    for a, b, c in zip(e, tr, te)
                ], key=lambda r: r['epoch'])
    return []


def first_epoch(history, metric, threshold):
    for row in history:
        if pd.notna(row[metric]) and row[metric] >= threshold:
            return float(row['epoch'])
    return np.nan


def infer_seed(rec, path):
    seed = first_present(rec, ('seed', 'random_seed', 'run_seed'))
    if seed is not None:
        try:
            return int(seed)
        except Exception:
            pass
    m = re.search(r'seed[_-]?(\d+)', path.stem, re.I)
    return int(m.group(1)) if m else np.nan


def infer_config(rec, path):
    return str(first_present(rec, ('config', 'config_name', 'run_name', 'experiment_name', 'name'), path.parent.name))


def infer_task(config, rec, path):
    text = ' '.join([config, str(first_present(rec, ('task', 'task_name', 'dataset'), '')), str(path)]).lower()
    if config in ADDITION_CONFIG_ORDER or 'addition' in text or 'modular_add' in text:
        return 'addition'
    if any(t in text for t in S5_TERMS) and any(t in text for t in CORRECTED_TERMS):
        return 's5_corrected'
    if 'sphere_wd1_corrected' in text:
        return 's5_corrected'
    return 'other'


def flatten_json(payload):
    if isinstance(payload, list):
        return [x for x in payload if isinstance(x, dict)]
    if not isinstance(payload, dict):
        return []
    for key in ('runs', 'results', 'records', 'per_run', 'seed_results'):
        value = payload.get(key)
        if isinstance(value, list) and value and all(isinstance(x, dict) for x in value):
            shared = {k: v for k, v in payload.items() if k != key and not isinstance(v, (list, dict))}
            return [{**shared, **x} for x in value]
    return [payload]


def normalize_record(rec, path, source_kind):
    config = infer_config(rec, path)
    hist = extract_history(rec)
    peak = first_present(rec, ('peak_test_acc', 'best_test_acc', 'max_test_acc'))
    final = first_present(rec, ('final_test_acc', 'last_test_acc', 'test_acc', 'test_accuracy'))
    if hist:
        vals = [r['test_acc'] for r in hist if pd.notna(r['test_acc'])]
        peak = max(vals) if peak is None and vals else peak
        final = vals[-1] if final is None and vals else final

    row = {
        'task': infer_task(config, rec, path),
        'config': config,
        'seed': infer_seed(rec, path),
        'model_type': first_present(rec, ('model_type', 'model', 'architecture')),
        'bounded_head': as_bool(first_present(rec, ('bounded_head', 'normalize_unembedding', 'normalized_unembedding', 'norm_unembedding'))),
        'tau': as_float(first_present(rec, ('tau', 'temperature', 'fixed_tau'))),
        'weight_decay': as_float(first_present(rec, ('weight_decay', 'wd', 'lambda'))),
        'lr': as_float(first_present(rec, ('lr', 'learning_rate'))),
        'max_epochs': as_float(first_present(rec, ('max_epochs', 'epochs', 'num_epochs'))),
        'stop_reason': first_present(rec, ('stop_reason', 'status')),
        'last_epoch': as_float(first_present(rec, ('last_epoch', 'final_epoch', 'epoch'), hist[-1]['epoch'] if hist else np.nan)),
        'peak_test_acc': acc(peak),
        'final_test_acc': acc(final),
        'source_file': str(path),
        'source_kind': source_kind,
        'source_mtime': path.stat().st_mtime if path.exists() else np.nan,
    }
    for thr in THRESHOLDS:
        lab = int(round(thr * 100))
        tr = first_present(rec, (f'train_epoch_{lab}', f'train_{lab}_epoch', f'train_onset_{lab}'))
        te = first_present(rec, (f'test_epoch_{lab}', f'test_{lab}_epoch', f'test_onset_{lab}'))
        tr = as_float(tr) if tr is not None else first_epoch(hist, 'train_acc', thr)
        te = as_float(te) if te is not None else first_epoch(hist, 'test_acc', thr)
        row[f'train_epoch_{lab}'] = tr
        row[f'test_epoch_{lab}'] = te
        row[f'gap_{lab}'] = te - tr if pd.notna(tr) and pd.notna(te) else np.nan
    for key in ('test_rise_epoch', 'train_acc_at_test_rise', 'test_acc_at_test_rise', 'train_delta_during_test_rise', 'test_delta_during_test_rise', 'local_train_test_corr'):
        row[key] = as_float(rec.get(key))
    return row


def roots_to_scan():
    roots = [p for p in PREFERRED_ROOTS if p.exists()]
    roots += [Path(p) for p in EXTRA_PATHS if Path(p).exists()]
    return list(dict.fromkeys(roots)) or [DRIVE_ROOT]


def candidate_files(roots):
    out = set()
    for root in roots:
        if root.is_file():
            out.add(root)
            continue
        for p in root.rglob('*.json'):
            if root != DRIVE_ROOT or any(h in str(p).lower() for h in PATH_HINTS):
                out.add(p)
        for p in root.rglob('*.csv'):
            if any(h in p.name.lower() for h in ('per_run', 'transition', 'result', 'summary', 'onset', 'grok', 's5', 'addition')):
                out.add(p)
    return sorted(out)


def collect():
    roots = roots_to_scan()
    files = candidate_files(roots)
    rows, manifest = [], []
    for path in files:
        before = len(rows)
        if path.suffix.lower() == '.json':
            try:
                payload = json.loads(path.read_text(encoding='utf-8'))
            except Exception:
                payload = None
            for rec in flatten_json(payload):
                rows.append(normalize_record(rec, path, 'json'))
        elif path.suffix.lower() == '.csv':
            try:
                df = pd.read_csv(path)
            except Exception:
                df = pd.DataFrame()
            if not df.empty and {'config', 'seed'} <= set(df.columns):
                for _, rec in df.iterrows():
                    rows.append(normalize_record(rec.dropna().to_dict(), path, 'csv'))
        manifest.append({'source_file': str(path), 'records_extracted': len(rows) - before})
    return roots, files, pd.DataFrame(rows), pd.DataFrame(manifest)


CORE = [
    'model_type', 'bounded_head', 'tau', 'weight_decay', 'lr', 'max_epochs', 'stop_reason', 'last_epoch',
    'peak_test_acc', 'final_test_acc', 'train_epoch_95', 'test_epoch_95', 'gap_95',
    'train_epoch_99', 'test_epoch_99', 'gap_99', 'train_epoch_100', 'test_epoch_100', 'gap_100',
    'test_rise_epoch', 'train_acc_at_test_rise', 'test_acc_at_test_rise',
    'train_delta_during_test_rise', 'test_delta_during_test_rise', 'local_train_test_corr'
]


def coalesce(group):
    group = group.copy()
    score = group[CORE].notna().sum(axis=1)
    bonus = group['stop_reason'].astype(str).str.contains('stable|complete|success', case=False, regex=True).astype(int) * 4
    group['_score'] = score + bonus
    group = group.sort_values(['_score', 'source_mtime'], ascending=False)
    out = {'task': group.iloc[0]['task'], 'config': group.iloc[0]['config'], 'seed': group.iloc[0]['seed']}
    for col in CORE:
        vals = group[col].dropna()
        out[col] = vals.iloc[0] if len(vals) else np.nan
    out['source_file'] = ' | '.join(dict.fromkeys(group['source_file'].astype(str)))
    out['source_kind'] = ' | '.join(dict.fromkeys(group['source_kind'].astype(str)))
    out['duplicate_records_merged'] = len(group)
    return pd.Series(out)


def dedupe(raw):
    raw = raw.copy()
    raw['_seed_key'] = raw['seed'].where(raw['seed'].notna(), raw['source_file'] + '::missing')
    # pandas-version compatible groupby/apply
    return raw.groupby(['task', 'config', '_seed_key'], dropna=False, sort=False).apply(coalesce).reset_index(drop=True)


def first_non_null(s):
    s = s.dropna()
    return s.iloc[0] if len(s) else np.nan


def summarize(df):
    rows = []
    for config, g in df.groupby('config', sort=False):
        rows.append({
            'config': config,
            'seeds': int(g['seed'].nunique(dropna=True)),
            'model_type': first_non_null(g['model_type']),
            'bounded_head': first_non_null(g['bounded_head']),
            'tau': first_non_null(g['tau']),
            'weight_decay': first_non_null(g['weight_decay']),
            'lr': first_non_null(g['lr']),
            'reached_95': int(g['test_epoch_95'].notna().sum()),
            'reached_99': int(g['test_epoch_99'].notna().sum()),
            'reached_100': int(g['test_epoch_100'].notna().sum()),
            'mean_test_95': g['test_epoch_95'].mean(), 'std_test_95': g['test_epoch_95'].std(), 'median_test_95': g['test_epoch_95'].median(),
            'mean_test_99': g['test_epoch_99'].mean(), 'std_test_99': g['test_epoch_99'].std(), 'median_test_99': g['test_epoch_99'].median(),
            'mean_test_100': g['test_epoch_100'].mean(), 'std_test_100': g['test_epoch_100'].std(), 'median_test_100': g['test_epoch_100'].median(),
            'mean_gap_95': g['gap_95'].mean(), 'std_gap_95': g['gap_95'].std(), 'median_gap_95': g['gap_95'].median(),
            'mean_gap_99': g['gap_99'].mean(), 'std_gap_99': g['gap_99'].std(), 'median_gap_99': g['gap_99'].median(),
            'mean_gap_100': g['gap_100'].mean(), 'std_gap_100': g['gap_100'].std(), 'median_gap_100': g['gap_100'].median(),
            'mean_peak_test_acc': g['peak_test_acc'].mean(),
            'min_peak_test_acc': g['peak_test_acc'].min(),
            'max_peak_test_acc': g['peak_test_acc'].max(),
            'min_last_epoch': g['last_epoch'].min(),
            'max_last_epoch': g['last_epoch'].max(),
        })
    return pd.DataFrame(rows)


def mean_sd(mean, sd, reached, n):
    if pd.isna(mean):
        return f'NA ({reached}/{n})'
    if pd.isna(sd):
        return f'{mean:.0f} ({reached}/{n})'
    return f'{mean:.0f} ± {sd:.0f} ({reached}/{n})'


def compact_addition(summary):
    if summary.empty:
        return pd.DataFrame()
    lookup = summary.set_index('config')
    rows = []
    for cfg in ADDITION_CONFIG_ORDER:
        if cfg not in lookup.index:
            rows.append({'config': cfg, 'residual': 'MISSING', 'head': 'MISSING', 'WD': np.nan, 'n': 0, 'test95_epoch': 'MISSING', 'gap95_epoch': 'MISSING', 'test99_epoch': 'MISSING', 'gap99_epoch': 'MISSING', 'mean_peak_test_acc': np.nan})
            continue
        r = lookup.loc[cfg]
        model = str(r['model_type']).lower()
        residual = 'Sphere' if 'spher' in model else 'LayerNorm'
        head = 'bounded' if bool(r['bounded_head']) else 'raw'
        n = int(r['seeds'])
        rows.append({
            'config': cfg, 'residual': residual, 'head': head, 'WD': r['weight_decay'], 'n': n,
            'test95_epoch': mean_sd(r['mean_test_95'], r['std_test_95'], int(r['reached_95']), n),
            'gap95_epoch': 'NA' if pd.isna(r['mean_gap_95']) else f"{r['mean_gap_95']:.0f} ± {r['std_gap_95']:.0f}" if pd.notna(r['std_gap_95']) else f"{r['mean_gap_95']:.0f}",
            'test99_epoch': mean_sd(r['mean_test_99'], r['std_test_99'], int(r['reached_99']), n),
            'gap99_epoch': 'NA' if pd.isna(r['mean_gap_99']) else f"{r['mean_gap_99']:.0f} ± {r['std_gap_99']:.0f}" if pd.notna(r['std_gap_99']) else f"{r['mean_gap_99']:.0f}",
            'mean_peak_test_acc': r['mean_peak_test_acc'],
        })
    return pd.DataFrame(rows)


PER_RUN = [
    'config', 'seed', 'model_type', 'bounded_head', 'tau', 'weight_decay', 'lr', 'max_epochs',
    'stop_reason', 'last_epoch', 'peak_test_acc', 'final_test_acc',
    'train_epoch_95', 'test_epoch_95', 'gap_95', 'train_epoch_99', 'test_epoch_99', 'gap_99',
    'train_epoch_100', 'test_epoch_100', 'gap_100', 'test_rise_epoch',
    'train_acc_at_test_rise', 'test_acc_at_test_rise', 'source_file'
]


def show(title, df, cols=None):
    print('\n' + '=' * 110)
    print(title)
    print('=' * 110)
    if df.empty:
        print('No rows found.')
    else:
        print((df[cols] if cols else df).to_string(index=False))


def md(df):
    if df.empty:
        return '_No rows found._'
    try:
        return df.to_markdown(index=False)
    except Exception:
        return '```\n' + df.to_string(index=False) + '\n```'


def main():
    if not DRIVE_ROOT.exists():
        try:
            from google.colab import drive
            drive.mount('/content/drive')
        except Exception as e:
            raise FileNotFoundError('Mount Google Drive or change DRIVE_ROOT.') from e

    roots, files, raw, manifest = collect()
    print('Search roots:')
    for r in roots:
        print(' -', r)
    print(f'Candidate files inspected: {len(files)}')
    print(f'Raw records extracted: {len(raw)}')
    if raw.empty:
        raise FileNotFoundError('No records found. Add folders/files to PREFERRED_ROOTS or EXTRA_PATHS.')

    all_runs = dedupe(raw)
    addition = all_runs[all_runs['config'].isin(ADDITION_CONFIG_ORDER)].copy()
    order = {c: i for i, c in enumerate(ADDITION_CONFIG_ORDER)}
    addition['_order'] = addition['config'].map(order)
    addition = addition.sort_values(['_order', 'seed']).drop(columns='_order')
    s5 = all_runs[all_runs['task'] == 's5_corrected'].sort_values(['config', 'seed']).copy()

    add_sum = summarize(addition)
    if not add_sum.empty:
        add_sum['_order'] = add_sum['config'].map(order)
        add_sum = add_sum.sort_values('_order').drop(columns='_order')
    s5_sum = summarize(s5)
    component = compact_addition(add_sum)

    notes = []
    missing = [c for c in ADDITION_CONFIG_ORDER if c not in set(addition['config'])]
    notes.append('Addition grid complete: all 8 configs found.' if not missing else 'Missing Addition configs: ' + ', '.join(missing))
    if not addition.empty:
        counts = addition.groupby('config')['seed'].nunique().to_dict()
        notes.append('Addition seed counts: ' + ', '.join(f'{k}={v}' for k, v in counts.items()))
    if s5.empty:
        notes.append('No corrected S5 runs found. Check paths/config names.')
    else:
        reached = int(s5['test_epoch_99'].notna().sum())
        total = int(s5['seed'].nunique(dropna=True))
        notes.append(f'Corrected S5 reached 99% in {reached}/{total} seeds.')
        if 0 < reached < total:
            notes.append('Corrected S5 is seed-bimodal at 99%. Report success count and successful-seed onset statistics together.')

    show('VALIDATION NOTES', pd.DataFrame({'note': notes}))
    show('ADDITION PER-RUN ONSETS', addition, PER_RUN)
    show('ADDITION CONFIGURATION SUMMARY', add_sum)
    show('ADDITION COMPACT COMPONENT TABLE', component)
    show('CORRECTED S5 PER-SEED ONSETS', s5, PER_RUN)
    show('CORRECTED S5 CONFIGURATION SUMMARY', s5_sum)

    s5_outcomes = pd.DataFrame()
    if not s5.empty:
        s5_outcomes = s5[['config', 'seed', 'peak_test_acc', 'final_test_acc', 'test_epoch_95', 'test_epoch_99', 'test_epoch_100', 'last_epoch', 'stop_reason']].copy()
        s5_outcomes['outcome_99'] = np.where(s5_outcomes['test_epoch_99'].notna(), 'reached_99', 'did_not_reach_99')
        show('CORRECTED S5 SEED OUTCOMES', s5_outcomes)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    all_runs.to_csv(OUTPUT_DIR / 'master_all_runs.csv', index=False)
    addition.to_csv(OUTPUT_DIR / 'addition_per_run.csv', index=False)
    add_sum.to_csv(OUTPUT_DIR / 'addition_summary.csv', index=False)
    component.to_csv(OUTPUT_DIR / 'addition_component_table.csv', index=False)
    s5.to_csv(OUTPUT_DIR / 's5_corrected_per_run.csv', index=False)
    s5_sum.to_csv(OUTPUT_DIR / 's5_corrected_summary.csv', index=False)
    s5_outcomes.to_csv(OUTPUT_DIR / 's5_seed_outcomes.csv', index=False)
    manifest.to_csv(OUTPUT_DIR / 'source_manifest.csv', index=False)

    report = [
        '# Grokking Rebuttal Master Summary',
        '',
        '**Onset definition.** First logged evaluation epoch at or above 95%, 99%, or 100%. Resolution is limited by evaluation cadence.',
        '',
        '**Gap definition.** Test onset minus train onset at the same threshold. Means use only successful seeds, so success counts must be reported with every mean.',
        '',
        '## Validation notes',
        *[f'- {x}' for x in notes],
        '', '## Addition component table', md(component),
        '', '## Addition per-run onsets', md(addition[PER_RUN] if not addition.empty else addition),
        '', '## Addition configuration summary', md(add_sum),
        '', '## Corrected S5 per-seed onsets', md(s5[PER_RUN] if not s5.empty else s5),
        '', '## Corrected S5 configuration summary', md(s5_sum),
        '', '## Corrected S5 seed outcomes', md(s5_outcomes),
    ]
    (OUTPUT_DIR / 'MASTER_SUMMARY.md').write_text('\n'.join(report), encoding='utf-8')

    print('\nSaved all outputs to:', OUTPUT_DIR)
    print('Main readable report:', OUTPUT_DIR / 'MASTER_SUMMARY.md')


if __name__ == '__main__':
    main()

In [ ]:
"""
S5: print grok onset epochs and train-test gaps, formatted to match the
addition component table.

Reads only the `summary` block of each saved JSON. No retraining, no
checkpoint loading, CPU only.

Following Table 4's convention, onset statistics are aggregated over
SUCCESSFUL seeds only; the success count is reported separately so the
failures stay visible.
"""

import os
import glob
import json

import numpy as np
import pandas as pd

SAVE_DIR = "/content/drive/MyDrive/Grokking_S5_Sweep_v2"

# Success is defined per threshold: a seed "reached 99" only if it actually
# recorded a 99% onset epoch. Mixing a 95% success criterion with 99% onset
# statistics silently averages over a different set of seeds than the success
# column reports.

# Print order. Anything found on disk but not listed still appears, at the end.
CONFIG_ORDER = [
    "layernorm_learned_control",
    "layernorm_uniform_control",
    "layernorm_uniform_no_pos",
    "sphere_wd1_corrected",
    "sphere_nowd_main",
    "sphere_nowd_lr_3e-4",
    "sphere_nowd_lr_3e-3",
    "sphere_nowd_tau_5",
    "sphere_nowd_tau_20",
    "sphere_nowd_wide_256",
    "sphere_nowd_deep_2layer",
    "sphere_nowd_uniform",
    "sphere_nowd_long_200k",
    # add once run:
    # "layernorm_nowd_control",
    # "layernorm_bounded_nowd_control",
]

FIELDS = [
    "config", "seed", "model_type", "uniform_attention",
    "use_positional_embedding", "wd", "lr", "tau",
    "d_model", "mlp_dim", "layers", "max_epochs", "status",
    "train_epoch_95", "train_epoch_99",
    "grok_epoch_95", "grok_epoch_99",
    "gap_95", "gap_99",
    "peak_test_acc", "final_train_acc", "final_test_acc",
]


# ----------------------------------------------------------------------
# LOAD
# ----------------------------------------------------------------------

rows = []
for path in sorted(glob.glob(os.path.join(SAVE_DIR, "**", "*.json"), recursive=True)):
    with open(path, "r") as f:
        blob = json.load(f)

    summary = blob.get("summary")
    if not summary or "grok_epoch_99" not in summary:
        continue

    row = {k: summary.get(k) for k in FIELDS}
    row["source"] = os.path.basename(path)
    rows.append(row)

if not rows:
    raise RuntimeError(f"No S5 summaries found under {SAVE_DIR}")

df = pd.DataFrame(rows)

for col in ["train_epoch_95", "train_epoch_99", "grok_epoch_95",
            "grok_epoch_99", "gap_95", "gap_99", "peak_test_acc"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["reached_95"] = df["grok_epoch_95"].notna()
df["reached_99"] = df["grok_epoch_99"].notna()

order = {name: i for i, name in enumerate(CONFIG_ORDER)}
df["_order"] = df["config"].map(lambda c: order.get(c, len(order)))
df = df.sort_values(["_order", "config", "seed"]).drop(columns="_order")


# ----------------------------------------------------------------------
# PER-RUN
# ----------------------------------------------------------------------

print("=" * 100)
print("PER-SEED  (S5)")
print("=" * 100)
print(
    df[[
        "config", "seed", "wd", "lr", "tau", "d_model", "layers",
        "train_epoch_99", "grok_epoch_95", "grok_epoch_99",
        "gap_95", "gap_99", "peak_test_acc",
        "reached_95", "reached_99", "status",
    ]].to_string(index=False)
)


# ----------------------------------------------------------------------
# PER-CONFIG, SUCCESSFUL SEEDS ONLY
# ----------------------------------------------------------------------

def mean_std(series):
    vals = series.dropna().to_numpy(dtype=float)
    if len(vals) == 0:
        return None, None
    return float(vals.mean()), float(vals.std(ddof=1)) if len(vals) > 1 else 0.0


agg_rows = []
for config, group in df.groupby("config", sort=False):
    ok95 = group[group["reached_95"]]
    ok99 = group[group["reached_99"]]
    n_total = len(group)

    entry = {
        "config": config,
        "seeds": n_total,
        "reached_95": f"{len(ok95)}/{n_total}",
        "reached_99": f"{len(ok99)}/{n_total}",
        "mean_peak_acc": float(group["peak_test_acc"].mean()),
    }
    for col in ["grok_epoch_95", "gap_95"]:
        m, s = mean_std(ok95[col])
        entry[f"{col}_mean"] = m
        entry[f"{col}_std"] = s
    for col in ["train_epoch_99", "grok_epoch_99", "gap_99"]:
        m, s = mean_std(ok99[col])
        entry[f"{col}_mean"] = m
        entry[f"{col}_std"] = s
    agg_rows.append(entry)

agg = pd.DataFrame(agg_rows)

print("\n" + "=" * 100)
print("PER-CONFIG  (onset stats over SUCCESSFUL seeds only, as in Table 4)")
print("=" * 100)
print(agg.to_string(index=False))


# ----------------------------------------------------------------------
# MARKDOWN, matching the addition table
# ----------------------------------------------------------------------

def fmt(mean, std):
    if mean is None or (isinstance(mean, float) and np.isnan(mean)):
        return "NA"
    return f"({mean:,.0f} \\pm {std:,.0f})"


print("\n" + "=" * 100)
print("MARKDOWN")
print("=" * 100)
print("| Config | n | Test 99 | Gap 99 | Reached 99 | Reached 95 | Mean peak acc |")
print("| ------ | -: | ------: | -----: | ---------: | ---------: | ------------: |")
for _, r in agg.iterrows():
    print(
        f"| {r['config']} | {r['seeds']} | "
        f"{fmt(r['grok_epoch_99_mean'], r['grok_epoch_99_std'])} | "
        f"{fmt(r['gap_99_mean'], r['gap_99_std'])} | "
        f"{r['reached_99']} | {r['reached_95']} | "
        f"{r['mean_peak_acc']:.4f} |"
    )


# ----------------------------------------------------------------------
# THE COMPARISON THAT MATTERS
# ----------------------------------------------------------------------

print("\n" + "=" * 100)
print("ACCELERATION CHECK: successful spherical seeds vs LayerNorm baseline")
print("=" * 100)

base = agg[agg["config"] == "layernorm_learned_control"]
sphere = agg[agg["config"] == "sphere_wd1_corrected"]

if len(base) and len(sphere):
    b = base.iloc[0]
    s = sphere.iloc[0]
    b_vals = df[(df["config"] == "layernorm_learned_control")
                & df["reached_99"]]["grok_epoch_99"].to_numpy(dtype=float)
    s_vals = df[(df["config"] == "sphere_wd1_corrected")
                & df["reached_99"]]["grok_epoch_99"].to_numpy(dtype=float)

    print(f"LayerNorm (learned attn, wd=1)  n={len(b_vals)}  "
          f"mean={b_vals.mean():,.0f}  sd={b_vals.std(ddof=1):,.0f}  "
          f"range=[{b_vals.min():,.0f}, {b_vals.max():,.0f}]")
    print(f"  seeds: {', '.join(f'{v:,.0f}' for v in sorted(b_vals))}")
    print(f"Bounded Sphere  (wd=1)         n={len(s_vals)}  "
          f"mean={s_vals.mean():,.0f}  sd={s_vals.std(ddof=1):,.0f}  "
          f"range=[{s_vals.min():,.0f}, {s_vals.max():,.0f}]")
    print(f"  seeds: {', '.join(f'{v:,.0f}' for v in sorted(s_vals))}")

    ratio = s_vals.mean() / b_vals.mean()
    print(f"\nonset ratio (sphere / baseline) = {ratio:.2f}x")

    # Range overlap, computed rather than asserted.
    lo = max(b_vals.min(), s_vals.min())
    hi = min(b_vals.max(), s_vals.max())
    if hi >= lo:
        span = max(b_vals.max(), s_vals.max()) - min(b_vals.min(), s_vals.min())
        print(f"ranges OVERLAP on [{lo:,.0f}, {hi:,.0f}] "
              f"({100 * (hi - lo) / span:.0f}% of the combined span)")
    else:
        print("ranges are DISJOINT")

    # Fraction of each sample falling inside the other's range.
    b_in = float(((b_vals >= s_vals.min()) & (b_vals <= s_vals.max())).mean())
    s_in = float(((s_vals >= b_vals.min()) & (s_vals <= b_vals.max())).mean())
    print(f"baseline seeds inside sphere range: {b_in:.0%}")
    print(f"sphere seeds inside baseline range: {s_in:.0%}")

    try:
        from scipy import stats
        t, p_t = stats.ttest_ind(s_vals, b_vals, equal_var=False)
        u, p_u = stats.mannwhitneyu(s_vals, b_vals, alternative="two-sided")
        print(f"\nWelch t-test      : t={t:.2f}  p={p_t:.3f}")
        print(f"Mann-Whitney U    : U={u:.1f}  p={p_u:.3f}")

        n_b = len(df[df["config"] == "layernorm_learned_control"])
        n_s = len(df[df["config"] == "sphere_wd1_corrected"])
        table = [[len(b_vals), n_b - len(b_vals)],
                 [len(s_vals), n_s - len(s_vals)]]
        _, p_f = stats.fisher_exact(table)
        print(f"Fisher exact (success rate): p={p_f:.3f}")
    except Exception as exc:
        print(f"\n(scipy unavailable, tests skipped: {exc})")

    print("\nInterpret from the numbers above, not from this script. To compare\n"
          "against Z_p, run the equivalent analysis on the addition logs; this\n"
          "script does not load them.")
else:
    print("Missing layernorm_learned_control or sphere_wd1_corrected.")


out = os.path.join(SAVE_DIR, "s5_grok_table.csv")
agg.to_csv(out, index=False)
df.to_csv(os.path.join(SAVE_DIR, "s5_grok_per_seed.csv"), index=False)
print("\nsaved:", out)